# Normalization scale as loss importance weight

After `zscore` or `std_only`, all variables have `std(x_norm) = 1`.
The scale is therefore **not** a variance correction — it is a **loss importance weight**:

```
loss_contribution_var  ∝  scale²  ×  MSE(x_norm_var)
```

Setting `scale=1` for all variables gives equal loss contribution.  
The table below lets you adjust priorities and immediately see what fraction  
of the total loss each variable owns.

**Rules of thumb:**
- Variables you want the VAE to reconstruct well → keep scale high (0.5–1.0)
- Chaotic / noisy diagnostics → lower scale (0.1–0.3)
- Static fields (lsm) → very low scale (0.05–0.1), they barely need learning
- `none`-normed cloud fractions have `std ≈ 0.4`, so without scale they already
  contribute less than zscore vars — a scale of 0.5 here is roughly equivalent
  to scale=0.2 for a zscore variable in terms of loss contribution

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ─────────────────────────────────────────────────────────────────────────────
# Dataset statistics (embedded)
# ─────────────────────────────────────────────────────────────────────────────
VAR_NAMES = [
    '10fg','10si','10u','10v','2d','2t','cbh','fog',
    'hcc','lcc','lsm','mcc','msl','q_850','sp','t_850',
    'tcc','tcw','tp','u_850','v_850','vis','w_850','z'
]
STD = np.array([
    4.3299451e+00, 2.9582992e+00, 3.2886195e+00, 3.3199751e+00,
    7.9684067e+00, 8.5489264e+00, 6.6793906e+03, 2.0027253e-01,
    4.2964888e-01, 4.5027056e-01, 3.9544085e-01, 4.0948805e-01,
    1.1974965e+03, 1.9972378e-03, 4.5984819e+03, 6.6869164e+00,
    4.1893700e-01, 7.3980894e+00, 4.5502096e-04, 7.3267422e+00,
    7.5560985e+00, 2.1279119e+04, 2.2331698e+00, 3.7608040e+03
])
vi = {v: i for i, v in enumerate(VAR_NAMES)}

# ─────────────────────────────────────────────────────────────────────────────
# Norm type per variable
# ─────────────────────────────────────────────────────────────────────────────
NORM_CONFIG = {
    '10si': 'zscore',   '10u':   'zscore',   '10v':   'zscore',
    '2d':   'zscore',   '2t':    'zscore',   'cbh':   'zscore',
    'hcc':  'none',     'lcc':   'none',     'lsm':   'none',
    'mcc':  'none',     'msl':   'zscore',   'q_850': 'std_only',
    'sp':   'zscore',   't_850': 'zscore',   'tcc':   'none',
    'tcw':  'std_only', 'tp':    'std_only', 'u_850': 'zscore',
    'v_850':'zscore',   'w_850': 'std_only', 'z':     'zscore',
}

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# *** EDIT THIS DICT TO CHANGE PRIORITIES ***
#
# scale = importance weight in [0, 1]
# loss_contribution ∝ scale²
#
# Reasoning per group:
#   1.0  → primary reconstruction target (temperature, pressure, wind)
#   0.8  → important but slightly secondary (humidity, geopotential)
#   0.5  → useful but not the main focus (cloud fractions, vertical wind)
#   0.1  → chaotic/noisy/static — still reconstructed but loss-downweighted
# ─────────────────────────────────────────────────────────────────────────────
IMPORTANCE = {
    # Primary atmospheric state
    '2t':    1.0,
    '2d':    1.0,
    't_850': 1.0,
    'msl':   1.0,
    'sp':    1.0,

    # Wind
    '10u':   1.0,
    '10v':   1.0,
    '10si':  1.0,
    'u_850': 1.0,
    'v_850': 1.0,
    'w_850': 0.5,   # vertical wind — noisier

    # Moisture / water
    'q_850': 0.8,
    'tcw':   0.8,
    'z':     1.0,

    # Precipitation — chaotic, extreme outliers dominate MSE
    'tp':    0.1,

    # Cloud fractions — diagnostic, bounded [0,1], norm=none so std≈0.4 already
    'hcc':   0.5,
    'lcc':   0.5,
    'mcc':   0.5,
    'tcc':   0.5,

    # Static / auxiliary
    'lsm':   0.1,   # binary static mask
    'cbh':   0.5,   # cloud base height
}

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Compute effective std after normalization (before scale)
# zscore / std_only → eff_std = 1.0  (by definition)
# none              → eff_std = raw σ
# ─────────────────────────────────────────────────────────────────────────────
EPS = 1e-10

def eff_std(norm: str, sig: float) -> float:
    return 1.0 if norm in ('zscore', 'std_only', 'minmax') else sig

rows = []
for var, scale in IMPORTANCE.items():
    norm    = NORM_CONFIG[var]
    sig     = STD[vi[var]]
    e_std   = eff_std(norm, sig)
    # loss contribution per variable = scale² × eff_std²
    contrib = (scale * e_std) ** 2
    rows.append(dict(
        variable  = var,
        norm      = norm,
        raw_std   = sig,
        eff_std   = e_std,
        scale     = scale,
        contrib   = contrib,
    ))

df = pd.DataFrame(rows).set_index('variable')
total = df['contrib'].sum()
df['loss_pct'] = 100 * df['contrib'] / total

pd.set_option('display.float_format', '{:.4f}'.format)
df[['norm','raw_std','eff_std','scale','contrib','loss_pct']]

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Summary: loss % per variable group
# ─────────────────────────────────────────────────────────────────────────────
groups = {
    'Primary state':    ['2t','2d','t_850','msl','sp','z'],
    'Wind':             ['10u','10v','10si','u_850','v_850','w_850'],
    'Moisture':         ['q_850','tcw'],
    'Precipitation':    ['tp'],
    'Cloud fractions':  ['hcc','lcc','mcc','tcc'],
    'Static/auxiliary': ['lsm','cbh'],
}

print(f"{'Group':<20} {'Loss %':>8}")
print("-" * 30)
for group, vars_ in groups.items():
    pct = df.loc[[v for v in vars_ if v in df.index], 'loss_pct'].sum()
    print(f"{group:<20} {pct:>7.2f}%")
print("-" * 30)
print(f"{'TOTAL':<20} {df['loss_pct'].sum():>7.2f}%")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Plot: loss % per variable
# ─────────────────────────────────────────────────────────────────────────────
COLOR_MAP = {
    'zscore':   '#3478c8',
    'std_only': '#2a9e6e',
    'none':     '#c87a30',
}

vars_   = df.index.tolist()
pcts    = df['loss_pct'].values
colors_ = [COLOR_MAP[df.loc[v, 'norm']] for v in vars_]

fig, ax = plt.subplots(figsize=(13, 4))
bars = ax.bar(vars_, pcts, color=colors_, edgecolor='none', alpha=0.87)

# annotate values
for bar, pct in zip(bars, pcts):
    if pct > 0.5:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f'{pct:.1f}%', ha='center', va='bottom', fontsize=8)

ax.set_ylabel('% of total loss')
ax.set_title('Loss contribution per variable  (scale² × eff_std²)', fontsize=11)
ax.set_xticklabels(vars_, rotation=45, ha='right', fontsize=9)
ax.grid(axis='y', alpha=0.25)

patches = [mpatches.Patch(color=c, label=k) for k, c in COLOR_MAP.items()]
ax.legend(handles=patches, fontsize=9, title='norm type')

plt.tight_layout()
plt.savefig('loss_importance.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Print ready-to-paste YAML
# ─────────────────────────────────────────────────────────────────────────────
print("normalizer:")
print("  _target_: dssml.data.helpers.normalizers.UnifiedNormalizer")
print("  default_norm_type: zscore")
print("  default_scale: 1.0")
print("  eps: 1.0e-6")
print("  overrides:")
for var, row in df.iterrows():
    print(f"    {var}:")
    print(f"      norm_type: {row['norm']}")
    print(f"      scale: {row['scale']}     # {row['loss_pct']:.1f}% of loss")